# Laboratory Work 1: Organizational Data Governance and Policy Automation
**Course:** Management of Organizational Data  
**Theme:** Organizational Data Governance Frameworks, Compliance, and Data Stewardship  
**Programming Language:** Python 3.10+  
**Tools:** `pandas`, `numpy`, `re` (regular expressions)

---

## Executive Summary
This laboratory work operationalizes key enterprise data governance principles:
1. **RACI Matrix Design (Part 1):** Structuring accountability and cross-functional roles for a customer data collection pipeline under Kazakhstan and GDPR regulations.
2. **Practical Python Compliance Engine (Part 2):** Building automated controls for completeness, regex validation (email & KZ IIN), age sanity checking, consent filtering, and name pseudonymization, followed by an automated Executive Audit Report.
3. **Critical Thinking Analysis (Part 3):** Evaluating Kazakhstan's Article 12 Data Localization mandates (On-Premises vs. Global Cloud) and contrasting Data Owner vs. Data Steward roles in master data duplicate resolution.


# Part 1: Theoretical and Strategic Framework (30% of Total Mark)

## Task 1: RACI Matrix Design

An e-commerce enterprise in Kazakhstan is introducing a new customer data collection pipeline. Below is the RACI Matrix defining operational and strategic governance roles across the 5 organizational functions:
- **Chief Data Officer (CDO):** Executive sponsor accountable for enterprise data strategy, governance policies, and executive alignment.
- **Lead Data Steward:** Operational guardian of data definitions, data quality metrics, and master records.
- **Data Engineer:** Technical builder who designs pipelines, writes validation/masking scripts, and manages databases.
- **Legal and Compliance Officer:** Regulatory expert ensuring adherence to the KZ Personal Data Protection Law and GDPR.
- **Marketing Manager:** Business stakeholder leveraging customer data for campaigns and revenue growth.

### RACI Definitions
- **R (Responsible):** The "Doer" who performs the work to complete the task.
- **A (Accountable):** The single decision-maker who holds ultimate ownership and veto power. (*Requirement: Exactly one 'A' per task*).
- **C (Consulted):** The subject matter expert consulted for bidirectional advice and requirements before execution.
- **I (Informed):** The stakeholder kept updated on progress, milestones, or outcomes (unidirectional).

---

### RACI Matrix Table

| # | Governance Task / Activity | Chief Data Officer (CDO) | Lead Data Steward | Data Engineer | Legal & Compliance Officer | Marketing Manager |
|:---:|:---|:---:|:---:|:---:|:---:|:---:|
| **1** | **Defining enterprise Data Quality rules for customer email and phone formats** | **A** | **R** | **C** | **I** | **C** |
| **2** | **Approving data access permissions for third-party marketing vendors** | **I** | **C** | **I** | **A** | **R** |
| **3** | **Implementing automated PII (Personally Identifiable Information) anonymization in data pipelines** | **A** | **C** | **R** | **C** | **I** |
| **4** | **Auditing cross-border data transfer compliance under GDPR and KZ PDP laws** | **I** | **C** | **C** | **A** | **I** |
| **5** | **Resolving duplicate customer master records in the database** | **I** | **A** | **R** | **I** | **C** |

---

### Detailed Justifications for Each Assignment

1. **Task 1: Defining enterprise Data Quality rules for customer email and phone formats**
   - **Accountable (A) – CDO:** The CDO owns enterprise-wide data governance standards and formally approves company data quality policies.
   - **Responsible (R) – Lead Data Steward:** The Data Steward actively drafts data definitions, acceptable format constraints, and regex standards.
   - **Consulted (C) – Data Engineer:** Consulted to confirm technical feasibility and database performance implications.
   - **Informed (I) – Legal & Compliance:** Informed of the baseline quality criteria for data intake.
   - **Consulted (C) – Marketing Manager:** Consulted to ensure data rules capture valid marketing channels without impeding customer acquisition.

2. **Task 2: Approving data access permissions for third-party marketing vendors**
   - **Accountable (A) – Legal and Compliance Officer:** Sharing personal data with third-party vendors involves statutory liability under GDPR and KZ PDP Law. Only Legal can approve cross-organization sharing agreements.
   - **Responsible (R) – Marketing Manager:** The Marketing Manager initiates the business request, provides vendor documentation, and justifies the campaign objective.
   - **Consulted (C) – Lead Data Steward:** Consulted to verify data sensitivity classification and enforce data minimization (least privilege).
   - **Informed (I) – Data Engineer:** Informed to configure API keys or IAM roles once legal approval is granted.
   - **Informed (I) – CDO:** Informed of vendor data sharing partnerships.

3. **Task 3: Implementing automated PII anonymization in data pipelines**
   - **Accountable (A) – CDO:** Accountable for ensuring the technical architecture enforces privacy-by-design standards across all data assets. *(Note: Data Engineer can also be designated Accountable for engineering pipeline delivery; exactly one 'A' is strictly maintained).*
   - **Responsible (R) – Data Engineer:** The Data Engineer writes, tests, and deploys the automated masking/pseudonymization algorithms inside the ETL/ELT pipelines.
   - **Consulted (C) – Lead Data Steward:** Consulted to specify which fields represent PII and the required masking formats.
   - **Consulted (C) – Legal & Compliance Officer:** Consulted to verify that the anonymization method satisfies statutory definitions under data protection laws.
   - **Informed (I) – Marketing Manager:** Informed that downstream marketing datasets will contain masked identifiers.

4. **Task 4: Auditing cross-border data transfer compliance under GDPR and KZ PDP laws**
   - **Accountable (A) – Legal and Compliance Officer:** Legally accountable for verifying cross-border data transfer mechanisms (e.g., standard contractual clauses, localization compliance) and issuing compliance verdicts.
   - **Responsible (R) – Legal and Compliance Officer:** Conducts the audit procedure, reviews contracts, and analyzes regulatory risks.
   - **Consulted (C) – Lead Data Steward:** Consulted to provide metadata, data catalogs, and data lineage documentation.
   - **Consulted (C) – Data Engineer:** Consulted to provide cloud region configurations, network topologies, server physical locations, and encryption protocols.
   - **Informed (I) – CDO & Marketing Manager:** Informed of audit findings, compliance status, and any required remediation.

5. **Task 5: Resolving duplicate customer master records in the database**
   - **Accountable (A) – Lead Data Steward:** In Master Data Management (MDM), the Data Steward owns record deduplication and establishes survivorship rules (which record becomes the golden record).
   - **Responsible (R) – Data Engineer:** The Data Engineer executes the database merge script, re-points foreign keys, and purges redundant rows.
   - **Consulted (C) – Marketing Manager:** Consulted to verify customer activity history so valuable transaction data is preserved.
   - **Informed (I) – CDO & Legal Officer:** Kept informed of master data quality metrics and audit trail records.


# Part 2: Practical Implementation in Python (70% of Total Mark)

## Task 2: Data Quality and Regulatory Compliance Engine

In this section, we build an automated Python engine using `pandas`, `numpy`, and regular expressions (`re`) that enforces enterprise data governance policies on dirty customer records (`raw_customer_data.csv`).

### Step 1: Dataset Setup
We load the sample raw dataset containing dirty customer records into a Pandas DataFrame and save it as `raw_customer_data.csv`.


In [1]:
import pandas as pd
import numpy as np
import re
import pprint

data = {
    "customer_id": [101, 102, 103, 104, 105, 106],
    "full_name": ["Arman Aliyev", "Elena Petrova", "John Doe", "Aisha Saparova", "Dmitry Ivanov", "Sanzhar N"],
    "email": ["arman@kbtu.kz", "elena.p@gmail.com", "invalid-email-at-com", "aisha@domain.org", np.nan, "sanzhar@company.kz"],
    "phone": ["+77015551234", "87071112233", "+77029998877", "12345", "+77053334455", "77080001122"],
    "age": [24, 17, 35, 140, 29, -5],
    "kz_iin": ["000115501234", "070420609876", "12345", "950812300111", "880315400222", "990101300000"],
    "consent_given": [True, True, True, False, True, False]
}

df = pd.DataFrame(data)
df.to_csv("raw_customer_data.csv", index=False)
df

,customer_id,full_name,email,phone,age,kz_iin,consent_given
0,101,Arman Aliyev,arman@kbtu.kz,+77015551234,24,000115501234,True
1,102,Elena Petrova,elena.p@gmail.com,87071112233,17,070420609876,True
2,103,John Doe,invalid-email-at-com,+77029998877,35,12345,True
3,104,Aisha Saparova,aisha@domain.org,12345,140,950812300111,False
4,105,Dmitry Ivanov,NaN,+77053334455,29,880315400222,True
5,106,Sanzhar N,sanzhar@company.kz,77080001122,-5,990101300000,False


### Step 2: Implement Governance Validation Functions

We implement automated governance checks across three primary control areas:
1. **Completeness Control:** Check for missing critical fields (`email`, `full_name`, `kz_iin`).
2. **Data Quality and Validity Checks:**
   - **Email Validation:** Verify email strings against regex `r"^[\w\.-]+@[\w\.-]+\.\w+$"`.
   - **Age Sanity Check:** Ensure age is within valid human lifespan ($18 \le \text{age} \le 120$). Flag out-of-bound values ($<0$ or $>120$) and identify underage minors ($<18$).
   - **Kazakhstan IIN Validation:** Ensure IIN is strictly 12 numeric digits using regex `r"^\d{12}$"`.
3. **Privacy and Compliance Policy Enforcement (GDPR & KZ PDP Law):**
   - **Consent Filtering:** Isolate records where `consent_given == False` so they are not processed for marketing.
   - **Data Anonymization (Pseudonymization):** Mask full names into initial + surname format (e.g., `"Arman Aliyev"` $\rightarrow$ `"A. Aliyev"`).


In [2]:
critical_fields = ["email", "full_name", "kz_iin"]
missing_counts = {col: int(df[col].isna().sum()) for col in critical_fields}

print("Completeness Control:")
for col, count in missing_counts.items():
    print(f"Missing {col}: {count}")

missing_email_mask = df["email"].isna()

Completeness Control:
Missing email: 1
Missing full_name: 0
Missing kz_iin: 0


In [3]:
EMAIL_REGEX = r"^[\w\.-]+@[\w\.-]+\.\w+$"
IIN_REGEX = r"^\d{12}$"

def validate_email(val):
    return bool(re.match(EMAIL_REGEX, str(val).strip())) if pd.notna(val) else False

def validate_iin(val):
    return bool(re.match(IIN_REGEX, str(val).strip())) if pd.notna(val) else False

invalid_email_format_mask = df["email"].notna() & (~df["email"].apply(validate_email))
invalid_age_mask = (df["age"] < 0) | (df["age"] > 120)
underage_minor_mask = (df["age"] >= 0) & (df["age"] < 18)
invalid_iin_mask = ~df["kz_iin"].apply(validate_iin)

print(f"Missing Email Count   : {missing_email_mask.sum()}")
print(f"Invalid Email Format  : {invalid_email_format_mask.sum()}")
print(f"Invalid Age Values    : {invalid_age_mask.sum()}")
print(f"Underage Minors (<18) : {underage_minor_mask.sum()}")
print(f"Invalid KZ IIN Strings: {invalid_iin_mask.sum()}")

Missing Email Count   : 1
Invalid Email Format  : 1
Invalid Age Values    : 2
Underage Minors (<18) : 1
Invalid KZ IIN Strings: 1


In [4]:
def anonymize_name(name: str) -> str:
    if not isinstance(name, str) or not name.strip():
        return ""
    parts = name.strip().split()
    return f"{parts[0][0].upper()}. {' '.join(parts[1:])}" if len(parts) > 1 else parts[0]

df["anonymized_name"] = df["full_name"].apply(anonymize_name)

consent_revoked_mask = ~df["consent_given"]
consent_revoked_df = df[consent_revoked_mask].copy()

print(f"Consent-Revoked Records Isolated: {len(consent_revoked_df)}")
df[["customer_id", "full_name", "anonymized_name", "consent_given"]]

Consent-Revoked Records Isolated: 2


,customer_id,full_name,anonymized_name,consent_given
0,101,Arman Aliyev,A. Aliyev,True
1,102,Elena Petrova,E. Petrova,True
2,103,John Doe,J. Doe,True
3,104,Aisha Saparova,A. Saparova,False
4,105,Dmitry Ivanov,D. Ivanov,True
5,106,Sanzhar N,S. N,False


### Step 3: Generate Automated Audit Report

We calculate the overall data quality metrics and generate an executive report:
- **Total Input Records Audited:** $6$
- **Consent Revoked (Isolated):** $2$ (Customers 104, 106)
- **Valid Records Approved:** $2$ (Customers 101, 102 passing email, age bound, and IIN checks)
- **Overall Data Quality Score:** $\frac{2}{6} \times 100\% = 33.33\%$


In [5]:
valid_records_mask = (
    (~missing_email_mask) & 
    (~invalid_email_format_mask) & 
    (~invalid_age_mask) & 
    (~invalid_iin_mask)
)

total_records = len(df)
consent_revoked_count = int(consent_revoked_mask.sum())
valid_approved_count = int(valid_records_mask.sum())
overall_quality_score = (valid_approved_count / total_records) * 100

summary_dictionary = {
    "Missing Email Count": int(missing_email_mask.sum()),
    "Invalid Email Format": int(invalid_email_format_mask.sum()),
    "Invalid Age Values": int(invalid_age_mask.sum()),
    "Invalid KZ IIN Strings": int(invalid_iin_mask.sum())
}

report = f"""==================================================
 DATA GOVERNANCE and COMPLIANCE AUDIT 
==================================================
Total Input Records Audited : {total_records}
Consent Revoked (Isolated) : {consent_revoked_count}
Valid Records Approved : {valid_approved_count}
--- Quality Control Failures ---
Missing Email Count : {summary_dictionary['Missing Email Count']}
Invalid Email Format : {summary_dictionary['Invalid Email Format']}
Invalid Age Values : {summary_dictionary['Invalid Age Values']}
Invalid KZ IIN Strings : {summary_dictionary['Invalid KZ IIN Strings']}
Overall Data Quality Score : {overall_quality_score:.2f}%
=================================================="""

print(report)

 DATA GOVERNANCE and COMPLIANCE AUDIT 
Total Input Records Audited : 6
Consent Revoked (Isolated) : 2
Valid Records Approved : 2
--- Quality Control Failures ---
Missing Email Count : 1
Invalid Email Format : 1
Invalid Age Values : 2
Invalid KZ IIN Strings : 1
Overall Data Quality Score : 33.33%


In [6]:
pprint.pprint(summary_dictionary)

{'Invalid Age Values': 2,
 'Invalid Email Format': 1,
 'Invalid KZ IIN Strings': 1,
 'Missing Email Count': 1}


In [7]:
audit_table = df.copy()
audit_table["missing_email"] = missing_email_mask
audit_table["invalid_email"] = invalid_email_format_mask
audit_table["invalid_age"] = invalid_age_mask
audit_table["invalid_iin"] = invalid_iin_mask
audit_table["quality_passed"] = valid_records_mask
audit_table["marketing_approved"] = valid_records_mask & audit_table["consent_given"]

audit_table[["customer_id", "anonymized_name", "quality_passed", "consent_given", "marketing_approved"]]

,customer_id,anonymized_name,quality_passed,consent_given,marketing_approved
0,101,A. Aliyev,True,True,True
1,102,E. Petrova,True,True,True
2,103,J. Doe,False,True,False
3,104,A. Saparova,False,False,False
4,105,D. Ivanov,False,True,False
5,106,S. N,False,False,False


# Part 3: Critical Thinking and Discussion Questions

---

### Question 1: Data Localization (Article 12 of Kazakhstan's PDP Law)
> **Question:** Article 12 of Kazakhstan's Personal Data Protection Law requires that personal data of KZ citizens be physically stored within the country. How does this law impact an enterprise deciding between storing user databases on-premises vs. using global public clouds (e.g., AWS, Microsoft Azure)?

#### Comprehensive Answer:

1. **The Legal Requirement (Article 12, Law No. 94-V):**
   - Article 12 of the Law of the Republic of Kazakhstan *"On Personal Data and their Protection"* mandates that the collection, recording, systematization, accumulation, and storage of personal data of Kazakhstani citizens must be performed using databases that are **physically located within the territory of the Republic of Kazakhstan**.
   - This principle is commonly known as **Data Localization**. Its purpose is to guarantee national data sovereignty, protect citizens' privacy, and ensure law enforcement and regulatory bodies can exercise jurisdictional authority.

2. **Impact on Choosing On-Premises Infrastructure:**
   - **Regulatory Compliance:** On-premises data centers (or local colocation facilities within Kazakhstan, e.g., in Almaty or Astana) guarantee strict, direct adherence to Article 12 because physical servers reside inside national borders.
   - **Operational Trade-offs:** High capital expenditure (CapEx) for server hardware, networking, cooling, physical security, and ongoing system administration. Scaling capacity requires months of hardware procurement and setup, and multi-region disaster recovery is complex and costly.

3. **Impact on Choosing Global Public Clouds (AWS, Microsoft Azure, Google Cloud):**
   - **The Compliance Barrier:** Global public cloud hyperscalers (such as AWS, Azure, and GCP) do not operate primary data center regions inside Kazakhstan (e.g., nearest AWS regions are in Frankfurt, Dublin, Bahrain, or Stockholm).
   - **Direct Violation Risk:** Storing the primary master database containing personal data of KZ citizens directly in an AWS S3 bucket, RDS instance, or Azure SQL database located overseas constitutes a direct violation of Article 12, exposing the enterprise to administrative fines, operational suspensions, or website blocking by state regulators.

4. **Strategic Architectural Solutions for Modern Enterprises:**
   - **A. Certified Local Cloud Providers (National IaaS):** Instead of on-premises hardware, enterprises can utilize local cloud providers that operate certified data centers physically located in Kazakhstan (e.g., Kazakhtelecom, QazCloud, PS.kz, Servercore Kazakhstan). This delivers cloud elasticity while satisfying 100% data localization.
   - **B. Hybrid Cloud Architecture:**
     - The **Primary Master Database** containing raw personal data (PII: names, IIN, phone numbers, addresses) is hosted strictly inside Kazakhstan (on-prem or local cloud).
     - **De-identification Before Cross-Border Transfer:** For advanced analytics, machine learning, or global CRM features hosted on AWS or Azure, data is first **anonymized, aggregated, or pseudonymized** locally. Under the law, truly anonymized data no longer constitutes personal data.
     - **Cross-Border Transfer Compliance:** If identifiable personal data must be transmitted abroad, the enterprise must obtain explicit customer consent and verify that the destination country provides adequate data protection laws or has bilateral agreements with Kazakhstan.

---

### Question 2: Data Stewardship (Data Owner vs. Data Steward)
> **Question:** If a database contains duplicate entries for the same customer (e.g., `"Arman Aliyev"` and `"A. Aliyev"`), explain the difference between what the Data Owner does and what the Data Steward does to resolve it.

#### Comprehensive Answer:

In enterprise data governance (such as the DAMA-DMBOK framework), **Data Owners** and **Data Stewards** operate at two complementary levels of authority: Strategic Accountability vs. Operational Execution.

```
+--------------------------------------------------------------+
|               DATA OWNER (Strategic / Accountable)           |
|  - Sets business policies and data definition standards      |
|  - Defines duplicate resolution & survivorship rules         |
|  - Authorizes record merges and deletes                      |
+--------------------------------------------------------------+
                              | (Authorizes & Mandates)
                              v
+--------------------------------------------------------------+
|              DATA STEWARD (Operational / Responsible)        |
|  - Identifies duplicate entries (Arman Aliyev vs A. Aliyev)  |
|  - Investigates matching keys (IIN, phone, transaction history)|
|  - Applies survivorship rules to construct the Golden Record |
|  - Works with Data Engineers to execute merge and prevent bugs |
+--------------------------------------------------------------+
```

#### 1. What the Data Owner Does (Accountability & Policy):
- **Role:** Usually a senior business executive (e.g., Chief Marketing Officer, VP of Customer Experience, or Head of Sales) who has business ownership of customer data.
- **Defines Survivorship Business Rules:** The Data Owner decides *which record wins* when duplicates collide. For example, the Data Owner dictates: *"If two customer records conflict, the record with the most recent transaction and a verified 12-digit IIN is the authoritative Master Record."*
- **Authorizes the Merge/De-duplication:** The Data Owner signs off on merging accounts and deprecating duplicate rows, accepting any commercial risk (such as merging loyalty points or transaction histories).
- **Allocates Resources:** Approves the master data management (MDM) budget and gives data teams the mandate to enforce data quality standards.

#### 2. What the Data Steward Does (Investigation & Operational Execution):
- **Role:** The day-to-day data custodian who understands both business definitions and data schemas.
- **Discovers and Investigates Duplicates:** The Data Steward detects the anomaly by querying the database. They investigate whether `"Arman Aliyev"` and `"A. Aliyev"` are truly the same person by cross-referencing secondary identifiers:
  - Do they share the same Kazakhstan IIN (`000115501234`)?
  - Do they share the same phone number or delivery address?
- **Executes the Survivorship Criteria:** Once confirmed as the same customer, the Data Steward applies the Data Owner's survivorship rules:
  - Sets `"Arman Aliyev"` as the canonical `full_name`.
  - Merges order history from both records into the master ID.
- **Coordinates Technical Remediation:** The Data Steward collaborates with the Data Engineer to run the database merge script, update foreign keys, and flag the duplicate record as inactive.
- **Root-Cause Analysis:** The Data Steward investigates why the duplicate occurred (e.g., absence of fuzzy matching at mobile app checkout) and updates the data validation backlog to prevent recurrence.

#### Summary Comparison Table:
| Dimension | Data Owner | Data Steward |
|:---|:---|:---|
| **Primary Question** | *"What should the policy be, and who approves it?"* | *"How do we identify, verify, and resolve this duplicate?"* |
| **Governance Role** | **Accountable (A)** | **Responsible (R)** |
| **Focus** | Strategic, commercial, policy, risk tolerance | Operational, investigative, data quality, technical execution |
| **Action on Duplicates** | Approves the survivorship rules and sign-off | Finds matching IINs, selects golden attributes, executes merge |
